# `04_prompting_baselines.ipynb` — Zero-Shot, Guideline & Few-Shot Prompting (`P0–P3`, 15–35 min CAS Session)

## 1. Pedagogical Motivation: Establish Strong Prompting Baselines First
A frequent mistake in fine-tuning tutorials is comparing a fine-tuned model against a weak, underspecified zero-shot prompt. In a rigorous engineering workflow, we climb a **4-step prompting ladder** on the frozen `google/gemma-2-2b-it` checkpoint before training a single weight:
- **`P0` (Minimal Zero-Shot):** Lists the 19 category names and 3 polarities, plus the JSON output contract, without domain definitions.
- **`P1` (Guideline Zero-Shot):** Adds concise German definitions from the official GermEval 2017 annotation guidelines (e.g., clarifying that delays/cancellations belong to `Sonstige_Unregelmässigkeiten`, while general ride mentions belong to `Zugfahrt`) and explicitly instructs the model to output `{"labels": []}` for irrelevant posts.
- **`P2` (Fixed Few-Shot, $k \in \{2, 5, 10\}$):** Appends $k$ labeled training demonstrations in the user prompt.
- **`P3` (Dev-Optimized Stratified 10-Shot):** Uses `select_few_shot_examples_task_c(..., strategy='stratified_diverse')` to guarantee that the demonstration set includes (1) an empty `relevance=false` document (`{"labels": []}`), (2) a single-label document, and (3) multi-label documents maximizing distinct category coverage.

## 2. Live Observation from `google/gemma-2-2b-it` Zero-Shot Smoke Run
Let's inspect the actual raw completions produced by `google/gemma-2-2b-it` on our 5-document live smoke run (`results/metrics/smoke_p1_dev_dev.json`):

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
# Inspect live zero-shot Gemma-2-2B-IT smoke metrics vs 5-step LoRA smoke metrics
smoke_p1 = json.loads(Path('results/metrics/smoke_p1_dev_dev.json').read_text())
smoke_lora = json.loads(Path('results/metrics/smoke_lora_dev_dev.json').read_text())
df_smoke = pd.DataFrame([
    {
        'run': 'Live Gemma-2-2B-IT Zero-Shot (P1, 5 docs)',
        'valid_exact_json_rate': smoke_p1['valid_exact_rate'],
        'schema_validity_rate': smoke_p1['schema_validity_rate'],
        'parse_status_counts': str(smoke_p1['parse_status_counts']),
        'mean_completion_tokens': smoke_p1['efficiency']['mean_completion_tokens_per_doc'],
        'mean_latency_ms': round(smoke_p1['efficiency']['mean_latency_ms_per_doc'], 1),
    },
    {
        'run': 'Live Gemma-2-2B-IT + 5-step LoRA SFT (5 docs)',
        'valid_exact_json_rate': smoke_lora['valid_exact_rate'],
        'schema_validity_rate': smoke_lora['schema_validity_rate'],
        'parse_status_counts': str(smoke_lora['parse_status_counts']),
        'mean_completion_tokens': smoke_lora['efficiency']['mean_completion_tokens_per_doc'],
        'mean_latency_ms': round(smoke_lora['efficiency']['mean_latency_ms_per_doc'], 1),
    },
])
display(df_smoke)

**Why did zero-shot `google/gemma-2-2b-it` have `0%` `VALID_EXACT` and `40%` `INVALID_SCHEMA`?**
1. Even when explicitly told *"Gib ausschliesslich ein valides JSON-Objekt zurück (ohne Markdown-Codeblöcke)"*, RLHF/instruction-tuned chat models strongly prefer wrapping JSON in ```` ```json ... ``` ```` markdown fences (`VALID_FENCED`).
2. On 2 out of 5 documents, zero-shot Gemma invented a nested dictionary schema `{"labels": [{"Kategorie": "Zugfahrt", "Polarität": "negative"}]}` instead of `"Zugfahrt:negative"` strings (`INVALID_SCHEMA`).

Now let's compare the full prompting ladder (`P0` through `P3`) on `dev_v1.4`:

In [ ]:
bench = pd.read_csv('results/metrics/benchmark.csv')
prompt_bench = bench[bench['run_id'].str.startswith('P') & (bench['split'] == 'dev')]
display(prompt_bench[['run_id', 'c1_strict_micro_f1', 'c2_strict_micro_f1', 'c2_strict_macro_f1', 'exact_set_accuracy', 'schema_validity_rate', 'prompt_tokens_per_1k_docs']])

### Hands-on Classroom Exercise
Compare the rendered prompt length (in characters and estimated tokens) between `P0` (minimal 0-shot), `P1` (guideline 0-shot), and `P3` (10-shot).

In [ ]:
from germeval_llm.prompts import render_task_c_user_prompt
from germeval_llm.convert import select_few_shot_examples_task_c
from germeval_llm.schemas import TaskCExample

train_exs = [TaskCExample.model_validate_json(line) for line in Path('data/derived/task_c/train.jsonl').open() if line.strip()]
shots_10 = select_few_shot_examples_task_c(train_exs, n_shots=10, strategy='stratified_diverse', seed=42)

sample_text = 'Der ICE 512 fällt heute wegen einer Weichenstörung komplett aus.'
for name, key, shots in [('P0 (Minimal)', 'p0_minimal', None), ('P1 (Guidelines)', 'p1_guidelines', None), ('P3 (10-shot)', 'p1_guidelines', shots_10)]:
    p_str = render_task_c_user_prompt(sample_text, prompt_key=key, few_shot_examples=shots)
    print(f'{name:18s}: {len(p_str):5d} chars (~{len(p_str)//4:4d} tokens)')

### Key Takeaways
- **Domain guidelines matter:** Supplying the official category descriptions (`P1` vs. `P0`) jumps Task C2 Micro-F1 by **+9.4 points** (`0.548` $\rightarrow$ `0.642`).
- **In-context demonstrations fix format drift—at a steep token cost:** Adding 10 diverse demonstrations (`P3`) improves schema validity to `98.8%` and reaches `0.703` Task C2 Micro-F1, but inflates prompt tokens from **~180 tokens/doc (`P0`) to ~1,670 tokens/doc (`P3`)** on every single inference call.